# 2. Gather posts

For each paper picked in notebook 1, pull all Altmetric `posts` rows that mention it. Bluesky / X / Reddit / blogs / news / Wikipedia / policy etc. all flow through here.

Inputs: `output/<RUN_ID>/papers.parquet` from notebook 1.
Outputs: `output/<RUN_ID>/posts.parquet`.

See `PIPELINE_PLAN.md` § S19 / S5 / S4. Dry-run cell precedes the real run.

## Setup

Set `RUN_ID` to a run created by notebook 1 (defaults to the most recent one).

In [1]:
import os
from pathlib import Path

os.environ.pop("GOOGLE_APPLICATION_CREDENTIALS", None)

REPO_ROOT = Path.cwd().parent.resolve()
OUTPUT_ROOT = REPO_ROOT / "altendor" / "output"

# Override RUN_ID here if you want a specific historical run instead of the latest.
RUN_ID: str | None = None
if RUN_ID is None:
    candidates = sorted([p.name for p in OUTPUT_ROOT.iterdir() if (p / "papers.parquet").exists()])
    if not candidates:
        raise FileNotFoundError("No run with papers.parquet found. Run notebook 1 first.")
    RUN_ID = candidates[-1]

OUT_DIR = OUTPUT_ROOT / RUN_ID
print(f"Using run id: {RUN_ID}\nOutput dir: {OUT_DIR}")

Using run id: 2026-06-28T21-34
Output dir: /home/automathis/Documents/Codebases/ICSSI-2026-Hackaton/altendor/output/2026-06-28T21-34


In [2]:
import pandas as pd
from google.cloud import bigquery

bq_client = bigquery.Client()

papers_df = pd.read_parquet(OUT_DIR / "papers.parquet")
ro_ids = papers_df["ro_id"].dropna().astype(str).tolist()
print(f"Loaded {len(papers_df)} papers \u2192 {len(ro_ids)} unique ro_ids")

Loaded 10 papers → 10 unique ro_ids


## Fetch posts

Server-side `UNNEST(research_outputs_ids)` explodes the posts so each (post, ro_id) pair becomes one row. A single post mentioning two of our papers appears twice — that's intentional, downstream stages key off `(post_id, ro_id)`.

**Dry-run first.**

In [3]:
from altendor.bigquery.preflight import preflight
from altendor.bigquery.queries import posts_by_research_output_ids

posts_sql, posts_cfg = posts_by_research_output_ids(ro_ids)
preflight(bq_client, posts_sql, job_config=posts_cfg)

Preflight: dry-run bytes masked by row-level security; upper bound from referenced tables:
  - altmetric_on_gbq.posts: 38649.85 MB  (256,107,288 rows)
Upper-bound scan: 38649.85 MB


38649847653

In [4]:
from altendor.sources.altmetric import fetch_posts_for_papers

posts_df = fetch_posts_for_papers(bq_client, ro_ids)
print(f"Fetched {len(posts_df)} post rows across {posts_df['ro_id'].nunique()} papers.")
posts_df.head(5)

Fetched 17218 post rows across 10 papers.


,post_id,ro_id,type,subtype,date,url,title,attention_source,retweet
0,587175429,34106210,wikipedia,NaN,2020-03-17 10:04:24+00:00,http://vi.wikipedia.org/?diff=prev&oldid=59522...,Thông tin sai lệch,"{'type': None, 'name': None, 'country': None}",False
1,269170630,4443094,blog,NaN,2019-03-01 23:16:13+00:00,https://replicationnetwork.com/2019/03/02/reed...,REED: Replications in Economics are Different ...,"{'type': 'blog', 'name': 'The Replication Netw...",False
2,91795577,4443094,blog,NaN,2016-01-20 07:00:00+00:00,http://www.npr.org/sections/money/2016/01/15/4...,The Experiment Experiment,"{'type': 'blog', 'name': 'SaveYourself.ca', 'c...",False
3,91846151,4443094,blog,NaN,2016-01-21 16:11:00+00:00,https://jcoynester.wordpress.com/2016/01/21/te...,Ten suggestions to the new associate editors o...,"{'type': 'blog', 'name': 'Quick Thoughts of J...",False
4,877080529,150735611,wikipedia,NaN,2026-06-27 23:33:23+00:00,http://sr.wikipedia.org/?diff=prev&oldid=31044...,Утицаји климатских промјена на људско здравље,"{'type': None, 'name': None, 'country': None}",False


## Per-platform stats

Quick distribution of post types. Bluesky (`bsky`) and Reddit (`rdt`) are the ones we'll deeply enrich (notebook 3); X (`tweet`), blogs, news (`msm`) keep `posts.title` as-is.

In [5]:
by_type = posts_df.groupby("type").size().rename("count").sort_values(ascending=False)
by_paper = posts_df.groupby("ro_id").size().rename("posts").sort_values(ascending=False)

print("Posts per platform:")
print(by_type.to_string())
print("\nPosts per paper:")
print(by_paper.to_string())

Posts per platform:
type
tweet          12593
msm             1714
rdt              683
blog             609
policy           602
wikipedia        434
fbwall           289
gplus             97
bluesky           66
podcast           52
qna               41
video             14
guideline         11
f1000              5
patent             5
peer_review        2
weibo              1

Posts per paper:
ro_id
80902343     3439
4443094      3348
15362282     3031
2345524      2285
34106210     1695
79872111     1394
150735611     742
33825242      693
4252201       307
21599659      284


## Write outputs

Persist `posts.parquet` and append to the run's `manifest.json`.

In [6]:
import json

posts_path = OUT_DIR / "posts.parquet"
posts_df.to_parquet(posts_path, index=False)

manifest_path = OUT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
manifest["stage_2_gather_posts"] = {
    "n_posts": int(len(posts_df)),
    "n_papers_with_posts": int(posts_df["ro_id"].nunique()),
    "by_platform": {str(k): int(v) for k, v in by_type.items()},
}
manifest.setdefault("output_files", []).append(str(posts_path.relative_to(REPO_ROOT)))
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
print(f"Wrote {posts_path}\nUpdated {manifest_path}")

Wrote /home/automathis/Documents/Codebases/ICSSI-2026-Hackaton/altendor/output/2026-06-28T21-34/posts.parquet
Updated /home/automathis/Documents/Codebases/ICSSI-2026-Hackaton/altendor/output/2026-06-28T21-34/manifest.json
